In [ ]:
# Install dependencies (required in Colab)
# Colab includes packages such as gradio and google-genai with their own
# pydantic/httpx constraints. Install compatible shared deps first, then
# install the SDK without letting older SDK metadata downgrade them.
%pip install -q --upgrade "pydantic>=2.0,<=2.12.3" "httpx>=0.28.1,<1.0" "attrs>=21.3.0" "python-dateutil>=2.8.0" "click>=8.0.0"
%pip install -q --upgrade --no-deps docu-devs-api-client

In [ ]:
import os
import json
from pathlib import Path

from docudevs import DocuDevsClient

# Set your API key:
# Option 1: Colab Secrets (recommended) - add DOCUDEVS_API_KEY in the 🔑 sidebar
# Option 2: Environment variable
# Option 3: Set directly below
try:
    from google.colab import userdata
    API_KEY = userdata.get("DOCUDEVS_API_KEY")
except Exception:
    API_KEY = os.getenv("DOCUDEVS_API_KEY", "your-api-key-here")

if not API_KEY or API_KEY == "your-api-key-here":
    raise ValueError("Set your DOCUDEVS_API_KEY - get one at https://docudevs.ai")

client = DocuDevsClient(token=API_KEY)
print("Connected to DocuDevs API")

# ---- Size knobs (kept small for the MINI tier) ----
RECORDS_TARGET = 2500          # records in the knowledge base (about 2-3k)
HOLDOUT_TARGET = 100           # held-out products that get classified (about 100)
PAGES_PER_CATEGORY = 14        # 100 products per search page, up to 42 search calls
MIN_LEAF_PRODUCTS = 20         # categories with fewer products are dropped
OFF_CATEGORIES = ["en:dairies", "en:snacks", "en:breakfast-cereals"]
OFF_COUNTRY = "en:finland"     # products sold in Finland; None = all countries
TARGET_DEPTH = [1, 2, 3]       # tree levels reported with each result
LLM_LEVEL = "MINI"
CLASSIFY_BUDGET_TOKENS = 600_000
HOLDOUT_EVAL_SAMPLE = 100
RUN_AGENT = False              # True starts the optional headless investigation
DELETE_CASE_AT_END = True

# Open Food Facts asks for a descriptive User-Agent with a contact.
# Replace the placeholder with your email or project URL before running.
CONTACT = "replace-with-your-email-or-project-url"
USER_AGENT = f"docudevs-python-examples/1.0 (+https://github.com/docudevs/python-examples; {CONTACT})"

DATA_DIR = Path("off-cache")   # local cache; never committed
DATA_DIR.mkdir(exist_ok=True)

In [ ]:
import random
import time
import urllib.error
import urllib.parse
import urllib.request
from collections import Counter, defaultdict

OFF_TAXONOMY_URL = "https://static.openfoodfacts.org/data/taxonomies/categories.json"
OFF_SEARCH_URL = "https://world.openfoodfacts.org/api/v2/search"
OFF_FIELDS = "code,product_name,generic_name,brands,quantity,ingredients_text,categories_tags"
PAGE_SIZE = 100
RATE_LIMIT_SECONDS = 6.5  # the search API allows 10 requests per minute per IP

TEXT_COLUMNS = ("name", "generic_name", "brand", "quantity", "ingredients")
FIELD_MAPPING = {
    "key": "ean",
    "text": {column: column for column in TEXT_COLUMNS},
    "label": "leaf",
    "metadata": {"brand": "brand"},
}
EMBEDDED_TEXT = {
    "template": "{name}\n{generic_name}\n{brand}\n{quantity}\n{ingredients}",
    "skipEmptyFields": True,
}


def single_chain(tag, parents):
    """Return [root, ..., tag] if every step up has exactly one parent, otherwise None."""
    if tag not in parents:
        return None
    chain = [tag]
    while parents[chain[-1]]:
        if len(parents[chain[-1]]) > 1:
            return None
        parent = parents[chain[-1]][0]
        if parent in chain or parent not in parents:
            return None
        chain.append(parent)
    return list(reversed(chain))


def most_specific_chain(tags, parents):
    """The deepest category of a product that has an unambiguous path to a root."""
    chains = [chain for chain in (single_chain(tag, parents) for tag in tags) if chain]
    return max(chains, key=lambda chain: (len(chain), chain[-1])) if chains else None


def parents_from_taxonomy(taxonomy):
    return {tag: list(entry.get("parents") or []) for tag, entry in taxonomy.items()}


def category_name(tag, taxonomy):
    names = (taxonomy.get(tag) or {}).get("name") or {}
    return names.get("en") or tag.split(":", 1)[-1].replace("-", " ").capitalize()


def parent_code(code):
    return code.rsplit(".", 1)[0] if "." in code else ""


def prefix(code, depth):
    return ".".join(code.split(".")[:depth])


def normalise_product(raw):
    code = str(raw.get("code") or "").strip()
    name = (raw.get("product_name") or "").strip()
    if not code or not name:
        return None
    return {
        "ean": code,
        "name": name[:200],
        "generic_name": (raw.get("generic_name") or "").strip()[:200],
        "brand": (raw.get("brands") or "").split(",")[0].strip()[:100],
        "quantity": (raw.get("quantity") or "").strip()[:100],
        "ingredients": (raw.get("ingredients_text") or "").strip()[:300],
        "categories_tags": list(raw.get("categories_tags") or []),
    }


def unique_products(raw_products):
    products = {}
    for raw in raw_products:
        product = normalise_product(raw)
        if product and product["ean"] not in products:
            products[product["ean"]] = product
    return list(products.values())


def chains_by_ean(products, parents):
    chains = {}
    for product in products:
        chain = most_specific_chain(product["categories_tags"], parents)
        if chain:
            chains[product["ean"]] = chain
    return chains


def prune_to_leaves(chains, min_products):
    """Drop products until every remaining label is a leaf with enough products.

    A category with fewer than min_products products is removed, and so is a product whose
    category is an ancestor of another kept category (it would be a non-leaf label).
    Repeat until nothing changes, because each removal can expose another.
    """
    kept = dict(chains)
    while True:
        counts = Counter(chain[-1] for chain in kept.values())
        ancestors = {tag for chain in kept.values() for tag in chain[:-1]}
        dropped = [
            ean
            for ean, chain in kept.items()
            if counts[chain[-1]] < min_products or chain[-1] in ancestors
        ]
        if not dropped:
            return kept
        for ean in dropped:
            del kept[ean]


def build_tree(chains, taxonomy):
    """Build dotted-code tree nodes from leaf chains. Returns (nodes, code_of_tag)."""
    parent_of = {}
    for chain in chains.values():
        for parent, child in zip(chain, chain[1:]):
            parent_of[child] = parent
    tags = {tag for chain in chains.values() for tag in chain}
    children = defaultdict(list)
    for child, parent in parent_of.items():
        children[parent].append(child)
    roots = sorted(
        (tag for tag in tags if tag not in parent_of),
        key=lambda tag: category_name(tag, taxonomy),
    )
    code_of = {}
    nodes = []
    queue = [(tag, str(index), None) for index, tag in enumerate(roots, start=1)]
    while queue:
        tag, code, parent = queue.pop(0)
        code_of[tag] = code
        nodes.append({"code": code, "parent": parent, "name": category_name(tag, taxonomy)})
        kids = sorted(children[tag], key=lambda kid: category_name(kid, taxonomy))
        queue.extend((kid, f"{code}.{index}", code) for index, kid in enumerate(kids, start=1))
    return nodes, code_of


def group_key(product):
    if product["brand"]:
        return "brand:" + product["brand"].lower()
    words = product["name"].lower().split()
    return "name:" + (words[0] if words else product["ean"])


def split_records_and_holdout(products, records_target, holdout_target, seed):
    """Move whole brand groups (or name-word groups) to the holdout, so no group is on both sides."""
    groups = defaultdict(list)
    for product in products:
        groups[group_key(product)].append(product)
    keys = sorted(groups)
    random.Random(seed).shuffle(keys)
    held = set()
    holdout = []
    for key in keys:
        if len(holdout) >= holdout_target:
            break
        if len(holdout) + len(groups[key]) <= holdout_target:
            held.add(key)
            holdout.extend(groups[key])
    rest = [product for key in sorted(groups) if key not in held for product in groups[key]]
    random.Random(seed + 1).shuffle(rest)
    return rest[:records_target], holdout


def record_row(product, leaf_code):
    row = {column: product[column] for column in TEXT_COLUMNS}
    row.update({"ean": product["ean"], "leaf": leaf_code})
    return row


def inline_row(product):
    row = {"key": product["ean"], "text": {c: product[c] for c in TEXT_COLUMNS if product[c]}}
    if product["brand"]:
        row["metadata"] = {"brand": product["brand"]}
    return row


def accuracy_by_level(results, truth_by_key, levels):
    scores = {}
    for depth in levels:
        right = 0
        for row in results:
            chosen = (row.get("chosen") or {}).get("code")
            if chosen and prefix(chosen, depth) == prefix(truth_by_key[row["key"]], depth):
                right += 1
        scores[depth] = (right, len(results))
    return scores


def confusable_pair(results, truth_by_key, leaves):
    """The most frequent sibling mix-up in the results, else the first pair of sibling leaves."""
    mistakes = Counter()
    for row in results:
        chosen = (row.get("chosen") or {}).get("code")
        truth = truth_by_key[row["key"]]
        if chosen and chosen != truth and parent_code(chosen) == parent_code(truth):
            mistakes[(truth, chosen)] += 1
    if mistakes:
        return mistakes.most_common(1)[0][0]
    by_parent = defaultdict(list)
    for leaf in leaves:
        by_parent[parent_code(leaf)].append(leaf)
    for siblings in by_parent.values():
        if len(siblings) >= 2:
            return siblings[0], siblings[1]
    return None


def alternative_leaf(code, leaves):
    siblings = [leaf for leaf in leaves if leaf != code and parent_code(leaf) == parent_code(code)]
    others = [leaf for leaf in leaves if leaf != code]
    return (siblings or others)[0]


def http_get_json(url, user_agent, attempts=4):
    request = urllib.request.Request(
        url, headers={"User-Agent": user_agent, "Accept": "application/json"}
    )
    for attempt in range(1, attempts + 1):
        try:
            with urllib.request.urlopen(request, timeout=60) as response:
                return json.loads(response.read().decode("utf-8"))
        except urllib.error.HTTPError as error:
            if error.code not in (429, 503) or attempt == attempts:
                raise
            print(f"Open Food Facts returned HTTP {error.code}; retrying in {30 * attempt}s")
            time.sleep(30 * attempt)


def load_taxonomy(cache_file):
    if not cache_file.exists():
        cache_file.write_text(json.dumps(http_get_json(OFF_TAXONOMY_URL, USER_AGENT)), encoding="utf-8")
    return json.loads(cache_file.read_text(encoding="utf-8"))


def search_products(categories, country, cache_file):
    if cache_file.exists():
        return json.loads(cache_file.read_text(encoding="utf-8"))
    raw = []
    for category in categories:
        for page in range(1, PAGES_PER_CATEGORY + 1):
            params = {
                "categories_tags": category,
                "fields": OFF_FIELDS,
                "page": page,
                "page_size": PAGE_SIZE,
                "sort_by": "unique_scans_n",
            }
            if country:
                params["countries_tags"] = country
            url = OFF_SEARCH_URL + "?" + urllib.parse.urlencode(params)
            products = http_get_json(url, USER_AGENT).get("products", [])
            raw.extend(products)
            print(f"{category} page {page}: {len(products)} products")
            time.sleep(RATE_LIMIT_SECONDS)
            if len(products) < PAGE_SIZE:
                break
    cache_file.write_text(json.dumps(raw), encoding="utf-8")
    return raw

In [ ]:
taxonomy = load_taxonomy(DATA_DIR / "categories.json")
parents = parents_from_taxonomy(taxonomy)
print(f"Taxonomy: {len(parents):,} categories")

def pool_for(country):
    tag = country.replace(":", "-") if country else "all"
    cache_file = DATA_DIR / f"search-{tag}-{'-'.join(c.split(':')[-1] for c in OFF_CATEGORIES)}-p{PAGES_PER_CATEGORY}.json"
    products = unique_products(search_products(OFF_CATEGORIES, country, cache_file))
    chains = chains_by_ean(products, parents)
    return products, chains, prune_to_leaves(chains, MIN_LEAF_PRODUCTS)

products, chains, kept = pool_for(OFF_COUNTRY)
if OFF_COUNTRY and len(kept) < RECORDS_TARGET + HOLDOUT_TARGET:
    print(f"Only {len(kept):,} usable products in {OFF_COUNTRY}; widening to all countries.")
    products, chains, kept = pool_for(None)
print(f"Downloaded {len(products):,} unique products; {len(chains):,} have a single category chain.")

In [ ]:
depths = Counter(len(chain) for chain in chains.values())
print("Depth of each product's most specific category (levels):")
for depth in sorted(depths):
    print(f"  {depth}: {depths[depth]:,}")

print("\nExamples:")
for product in products[:3]:
    chain = chains.get(product["ean"])
    path = " > ".join(category_name(tag, taxonomy) for tag in chain) if chain else "(no single chain)"
    print(f"- {product['name']} | brand: {product['brand'] or '-'} | {product['quantity'] or '-'}")
    print(f"  {path}")

In [ ]:
kept_chains = kept
nodes, code_of = build_tree(kept_chains, taxonomy)
leaves = sorted({code_of[chain[-1]] for chain in kept_chains.values()})
leaf_of = {ean: code_of[chain[-1]] for ean, chain in kept_chains.items()}
max_depth = max(len(code.split(".")) for code in leaves)
print(f"Tree: {len(nodes)} categories, {len(leaves)} leaves, depth {max_depth}")
print(f"Products kept: {len(kept_chains):,}")
for code in leaves[:8]:
    count = sum(1 for leaf in leaf_of.values() if leaf == code)
    name = next(node["name"] for node in nodes if node["code"] == code)
    print(f"  {code:8} {name} ({count})")

In [ ]:
kept_products = [product for product in products if product["ean"] in kept_chains]
records_pool, holdout_pool = split_records_and_holdout(
    kept_products,
    RECORDS_TARGET,
    HOLDOUT_TARGET,
    seed=20261011,
)
records_leaves = {leaf_of[product["ean"]] for product in records_pool}
holdout = [product for product in holdout_pool if leaf_of[product["ean"]] in records_leaves]

records_rows = [record_row(product, leaf_of[product["ean"]]) for product in records_pool]
truth_by_key = {product["ean"]: leaf_of[product["ean"]] for product in holdout}
records_by_ean = {row["ean"]: row for row in records_rows}
print(f"Records: {len(records_rows):,} products in {len(records_leaves)} leaves")
print(f"Holdout: {len(holdout)} products (leaves present in the records)")

In [ ]:
from docudevs.models.cases_controller_create_case_request import CasesControllerCreateCaseRequest


def get_case_id(response):
    """Extract the case ID from a 200 or 201 response."""
    if response.parsed is not None:
        return response.parsed.id
    return json.loads(response.content.decode())["id"]


created = await client.create_case(
    body=CasesControllerCreateCaseRequest(name="Open Food Facts products (notebook)")
)
if created.status_code not in (200, 201):
    raise RuntimeError(f"Error creating case: {created.status_code} {created.content.decode()}")
case_id = get_case_id(created)
print(f"Created case {case_id}")

dataset = await client.set_dataset(
    case_id,
    FIELD_MAPPING,
    embedded_text=EMBEDDED_TEXT,
    index_settings={"topK": 10, "hybrid": False},
    classify_defaults={"targetDepth": TARGET_DEPTH},
)
print("Dataset configured")

In [ ]:
imported = await client.bulk_upsert_records(case_id, records_rows)
final = await client.wait_for_import(case_id, imported["importId"], timeout=1800, poll_interval=5.0)
if final["status"] != "COMPLETED":
    raise RuntimeError(f"Import ended with {final['status']}")
print({key: final.get(key) for key in ("rowsReceived", "rowsRejected", "recordsAdded")})

In [ ]:
uploaded = await client.upload_tree(case_id, nodes, name="Open Food Facts categories")
version_id = uploaded["treeVersionId"]
await client.activate_tree(case_id, version_id, wait=True, timeout=1800)
print(f"Tree version {version_id} is active with {len(leaves)} leaves")

In [ ]:
classify_config = {
    "llmLevel": LLM_LEVEL,
    "concurrency": 4,
    "maxTotalTokens": CLASSIFY_BUDGET_TOKENS,
    "targetDepth": TARGET_DEPTH,
}
accepted = await client.classify(case_id, rows=[inline_row(p) for p in holdout], config=classify_config)
job_id = int(accepted["jobId"])
job = await client.wait_for_classify(case_id, job_id, timeout=3600, poll_interval=5.0)
print(f"Classify job {job_id}: {job['status']}", job.get("counts"))

results = [row async for row in client.get_classify_results(case_id, job_id)]
declined = sum(1 for row in results if not row.get("chosen"))
print(f"{len(results)} rows, {declined} declined or failed")

levels = sorted(set(TARGET_DEPTH) | {max_depth})
for depth, (right, total) in accuracy_by_level(results, truth_by_key, levels).items():
    print(f"Level {depth}: {right}/{total} correct ({right / total:.0%})")

In [ ]:
flagged = [row for row in results if (row.get("review") or {}).get("required")]
print(f"{len(flagged)} of {len(results)} rows need review")

for row in flagged[:5]:
    chosen = row.get("chosen") or {}
    print(f"\nRow {row['rowId']}: {row['key']} ({truth_by_key[row['key']]} is the historical label)")
    print(f"  Chosen: {' > '.join(chosen.get('path', [])) or 'declined'}")
    print(f"  Reasons: {', '.join(row['review']['reasons'])}")
    if row.get("alternativeBestFit"):
        print(f"  Model's alternative: {row['alternativeBestFit'].get('reason')}")
    for ean in row.get("evidenceKeys", [])[:3]:
        evidence = records_by_ean.get(ean, {})
        print(f"  Evidence: {evidence.get('name', ean)} -> {evidence.get('leaf', '?')}")

In [ ]:
async def wait_for_label_write(review):
    write = review.get("labelWrite") or {}
    if write.get("importId") is not None:
        final = await client.wait_for_import(case_id, write["importId"], timeout=1800, poll_interval=2.0)
        if final["status"] != "COMPLETED":
            raise RuntimeError(f"Label write ended {final['status']}")


answered = [row for row in results if row.get("status") == "ok" and row.get("chosen")]
accept_row, correct_row = answered[0], answered[1]

accepted_review = await client.review_classify(case_id, job_id, accept_row["rowId"], "accept")
await wait_for_label_write(accepted_review)
accepted_label = (await client.get_record(case_id, accept_row["key"])).get("label")
print(f"Accepted {accept_row['key']}: label is {accepted_label}")

corrected_code = alternative_leaf(correct_row["chosen"]["code"], leaves)
corrected_review = await client.review_classify(
    case_id,
    job_id,
    correct_row["rowId"],
    "correct",
    code=corrected_code,
    reason="Reviewed in the notebook: the product name points to a neighbouring category.",
)
await wait_for_label_write(corrected_review)
corrected_label = (await client.get_record(case_id, correct_row["key"])).get("label")
print(f"Corrected {correct_row['key']}: {correct_row['chosen']['code']} -> {corrected_label}")

In [ ]:
pair = confusable_pair(results, truth_by_key, leaves)
if pair is None:
    raise RuntimeError("No sibling categories to write a rule for")
truth_code, predicted_code = pair
truth_name = next(node["name"] for node in nodes if node["code"] == truth_code)
predicted_name = next(node["name"] for node in nodes if node["code"] == predicted_code)

rule_text = (
    f"Between '{truth_name}' and '{predicted_name}': file a product under '{truth_name}' "
    f"when its name or generic name describes that product type. Use '{predicted_name}' only "
    "when the product name names that type."
)
entry = await client.add_memo_entry(
    case_id,
    "instruction",
    {"kind": "subtree", "code": parent_code(truth_code) or truth_code},
    rule_text,
)
approved = await client.approve_memo_entry(case_id, entry["id"])
print(f"Approved memo entry {entry['id']} (status {approved.get('status')})")
print(rule_text)

In [ ]:
variants = [
    {
        "name": "without-memo",
        "config": {"llmLevel": LLM_LEVEL, "targetDepth": TARGET_DEPTH, "memo": {"use": "none"}},
    },
    {"name": "with-memo"},
]
started = await client.start_holdout_eval(
    case_id,
    variants,
    sample_size=HOLDOUT_EVAL_SAMPLE,
    seed=7,
    config={"llmLevel": LLM_LEVEL, "targetDepth": TARGET_DEPTH},
)
finished = await client.wait_for_eval(case_id, started["evalId"], timeout=3600, poll_interval=5.0)

report = finished.get("report") or {}
print("Grouping:", (report.get("sample") or {}).get("grouping"))
for variant in report.get("variants") or []:
    target = ((variant.get("score") or {}).get("targetAccuracy")) or {}
    print(f"{variant.get('name')}: target accuracy {target.get('all')} over {target.get('nAll')} rows")
print("Paired differences:")
print(json.dumps(report.get("comparisons"), indent=2)[:2000])

In [ ]:
if RUN_AGENT:
    started_run = await client.start_agent_run(
        case_id,
        "investigate-flagged",
        params={"job_id": job_id, "max_rows": 3},
        budget={"max_total_tokens": 20_000},
        origin_job_id=job_id,
    )
    run = await client.wait_for_agent_run(case_id, started_run["runId"], timeout=1800, poll_interval=5.0)
    print(run["status"], "| tokens:", run.get("tokensUsed"), "| note:", run.get("error"))
    print((run.get("report") or {}).get("summary"))
    for action in (await client.list_agent_actions(case_id)).get("items", []):
        print(action.get("status"), "-", action.get("summary"))
else:
    print("Set RUN_AGENT = True to start a headless investigation (20k token budget).")

In [ ]:
if DELETE_CASE_AT_END:
    await client.delete_case(case_id)
    print(f"Deleted case {case_id}")
else:
    print(f"Kept case {case_id}")